# FMI CloudCast → GK2A 겨울 파인튜닝 (Step 1 게이트)

**목적**: 겨울(1월) 운량 나우캐스트에서 광류 이류(M1)가 실패하는 구간을 DL로 보완할 수 있는지 판정.

**준비물** (Google Drive `MyDrive/nwp_dl/`에 업로드):
- `fmi_cloudcast_unet.tar.gz` (사전학습 가중치, 340MB)
- `dataset/2025-12-*.npz` + `dataset/2026-01-*.npz` (dl_dataset.py 산출 — 파일명은 YYYYMMDD.npz)

**규약**: 12월=학습, 1월=검증(학습에 절대 미사용). 런타임: GPU(T4) 선택 후 전체 실행.
라이선스 미명시 자료이므로 개인 연구 용도로만 사용.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/nwp_dl'
import os, glob, tarfile, json, math, datetime as dt
import numpy as np
import tensorflow as tf
print('TF', tf.__version__, 'GPU:', tf.config.list_physical_devices("GPU"))

In [ ]:
# 1) 가중치 압축 해제 + 로드 (custom loss 없이)
MODEL_DIR = '/content/fmi_model'
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR, exist_ok=True)
    with tarfile.open(f'{BASE}/fmi_cloudcast_unet.tar.gz') as t:
        t.extractall(MODEL_DIR)
# SavedModel 루트 탐색
cands = [r for r, d, f in os.walk(MODEL_DIR) if 'saved_model.pb' in f]
print('SavedModel:', cands)
model = tf.keras.models.load_model(cands[0], compile=False)
IN_SHAPE = model.inputs[0].shape
N_CH = int(IN_SHAPE[-1])
print('입력:', IN_SHAPE, '→ 채널', N_CH)  # 기대: 4(hist)+1(sun)+12(leadtime onehot)=17

In [ ]:
# 2) 데이터 로드 (uint8 0..100, 255=결측 → float 0..1)
def load_month(pattern):
    stamps, frames = [], []
    for f in sorted(glob.glob(f'{BASE}/dataset/{pattern}.npz')):
        z = np.load(f)
        stamps += list(z['stamps'])
        frames.append(z['frames'])
    fr = np.concatenate(frames)
    return np.array(stamps), fr

st_tr, fr_tr = load_month('202512??')
st_va, fr_va = load_month('202601??')
print('학습(12월):', fr_tr.shape, ' 검증(1월):', fr_va.shape)
IDX_TR = {s: i for i, s in enumerate(st_tr)}
IDX_VA = {s: i for i, s in enumerate(st_va)}

In [ ]:
# 3) 태양고도 채널 (FMI sun=True) — 512² 위경도는 GK2A KO LCC에서 근사
#    (풀 정확도 불필요: 모델 입력용 정규화 sin(elev) 0..1)
import numpy as np
# KO 격자 모서리 위경도 근사(LCC 30/60, 38N/126E, ±899km) — 사전 계산값
LATS = np.linspace(46.0, 29.7, 512)[:, None] * np.ones((1, 512))
LONS = np.ones((512, 1)) * np.linspace(113.0, 139.5, 512)[None, :]

def sun_channel(stamp):
    t = dt.datetime.strptime(stamp, '%Y%m%d%H%M')
    doy = t.timetuple().tm_yday
    decl = -23.44 * math.cos(math.radians(360/365*(doy+10)))
    hour = t.hour + t.minute/60
    ha = (hour*15 - 180) + LONS  # 시간각(deg), UTC 기준
    sin_el = (np.sin(np.radians(LATS))*math.sin(math.radians(decl)) +
              np.cos(np.radians(LATS))*math.cos(math.radians(decl))*np.cos(np.radians(ha)))
    return ((sin_el + 1) / 2).astype(np.float32)  # 0..1

In [ ]:
# 4) 학습 샘플 생성기 — X=[hist4 | sun(target) | onehot(lc)] , y=target
STEP = 10  # 분
N_HIST, N_LC = 4, max(0, N_CH - 5)
print('leadtime 채널 수:', N_LC)

def seq_ok(idx, stamps_set, s0, k):
    t0 = dt.datetime.strptime(s0, '%Y%m%d%H%M')
    need = [(t0 - dt.timedelta(minutes=STEP*i)).strftime('%Y%m%d%H%M') for i in range(N_HIST-1, -1, -1)]
    need.append((t0 + dt.timedelta(minutes=STEP*(k+1))).strftime('%Y%m%d%H%M'))
    return all(n in stamps_set for n in need), need

def make_sample(stamps, frames, idx, s0, k):
    ok, need = seq_ok(idx, idx.keys(), s0, k)
    if not ok:
        return None
    arrs = [frames[idx[n]] for n in need]
    if any((a == 255).mean() > 0.1 for a in arrs):
        return None
    hist = [np.where(a == 255, 50, a).astype(np.float32)/100.0 for a in arrs[:N_HIST]]
    y = np.where(arrs[-1] == 255, 50, arrs[-1]).astype(np.float32)/100.0
    ch = hist + [sun_channel(need[-1])]
    if N_LC:
        oh = np.zeros((512, 512, N_LC), np.float32); oh[..., min(k, N_LC-1)] = 1.0
        X = np.concatenate([np.stack(ch, -1), oh], -1)
    else:
        X = np.stack(ch, -1)
    return X, y[..., None]

def gen(stamps, frames, idx, n_lead, shuffle=True):
    keys = list(idx.keys())
    while True:
        order = np.random.permutation(len(keys)) if shuffle else range(len(keys))
        for i in order:
            k = np.random.randint(n_lead)
            s = make_sample(stamps, frames, idx, keys[i], k)
            if s is not None:
                yield s

N_LEAD_TRAIN = N_LC if N_LC else 12
sig = (tf.TensorSpec((512, 512, N_CH), tf.float32), tf.TensorSpec((512, 512, 1), tf.float32))
ds_tr = tf.data.Dataset.from_generator(lambda: gen(st_tr, fr_tr, IDX_TR, N_LEAD_TRAIN), output_signature=sig).batch(4).prefetch(2)
ds_va = tf.data.Dataset.from_generator(lambda: gen(st_va, fr_va, IDX_VA, N_LEAD_TRAIN, shuffle=False), output_signature=sig).batch(4).take(200)

In [ ]:
# 5) 손실(bc+l1, FMI bcl1 방식) + 파인튜닝
def bcl1(y_true, y_pred):
    bc = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred), axis=-1)
    return bc + l1

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=bcl1, metrics=['mae'])
hist = model.fit(ds_tr, steps_per_epoch=1500, epochs=3, validation_data=ds_va)
# T4 기준 대략 2~4시간. 중간에 세션이 끊기면 epochs를 낮춰 재실행.

In [ ]:
# 6) 저장 → Drive (로컬 M4 추론용)
out = f'{BASE}/gk2a_finetuned'
model.save(out)
!cd {BASE} && tar czf gk2a_finetuned.tar.gz gk2a_finetuned && ls -lh gk2a_finetuned.tar.gz
print('완료 — gk2a_finetuned.tar.gz를 PC의 kpx-model-charts/dl/ 에 내려받으세요')

In [ ]:
# 7) (선택) 1월 사례 눈검증 — 임의 시각 +60분 예측 vs 실제
import matplotlib.pyplot as plt
s0 = st_va[len(st_va)//2]
smp = make_sample(st_va, fr_va, IDX_VA, s0, 5)
if smp:
    X, y = smp
    p = model.predict(X[None])[0, ..., 0]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, img, t in zip(ax, [X[..., N_HIST-1], p, y[..., 0]], ['입력(t)', '예측 +60분', '실제']):
        a.imshow(img, cmap='gray', vmin=0, vmax=1); a.set_title(t); a.axis('off')
    plt.show()